In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
# Alligator cracking segmentation

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/AligatorCracking"
output_folder = "../../frames/segmented/AlligatorCracking"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0


# Segment alligator cracks
def segment_alligator(gray):

    # Step 1: Light smoothing
    blur = cv2.GaussianBlur(gray, (3,3), 0)

    # Step 2: Blackhat enhancement
    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (9,9)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        kernel
    )

    # Step 3: Binary threshold
    _, binary = cv2.threshold(
        blackhat,
        20,
        255,
        cv2.THRESH_BINARY
    )

    # Step 4: Remove tiny noise
    open_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3,3)
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        open_kernel,
        iterations=1
    )

    # Step 5: Moderate connection
    # Prevent whole-image merging
    close_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5,5)
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        close_kernel,
        iterations=1
    )

    return binary


# Merge nearby boxes
def merge_boxes(boxes, distance=35):

    merged_boxes = []

    while boxes:

        x1, y1, x2, y2 = boxes.pop(0)

        merged = True

        while merged:

            merged = False

            remove_indices = []

            for i, (xx1, yy1, xx2, yy2) in enumerate(boxes):

                # Check nearby distance
                if (
                    abs(xx1 - x2) < distance or
                    abs(x1 - xx2) < distance
                ) and (
                    abs(yy1 - y2) < distance or
                    abs(y1 - yy2) < distance
                ):

                    x1 = min(x1, xx1)
                    y1 = min(y1, yy1)
                    x2 = max(x2, xx2)
                    y2 = max(y2, yy2)

                    remove_indices.append(i)

                    merged = True

            for index in sorted(remove_indices, reverse=True):
                boxes.pop(index)

        merged_boxes.append([x1, y1, x2, y2])

    return merged_boxes


# Detect alligator crack regions
def detect_alligator_regions(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    # Small crack cell detection
    for c in contours:

        area = cv2.contourArea(c)

        if 100 < area < 2500:

            x, y, w, h = cv2.boundingRect(c)

            aspect_ratio = w / float(h + 1e-5)

            # Ignore long lines
            if 0.4 < aspect_ratio < 3.5:

                boxes.append([x, y, x+w, y+h])

    # Merge nearby boxes moderately
    merged_boxes = merge_boxes(boxes)

    # Draw medium-sized regions only
    for box in merged_boxes:

        x1, y1, x2, y2 = box

        width = x2 - x1
        height = y2 - y1

        area = width * height

        # Prevent huge full-image box
        if 3000 < area < 60000:

            cv2.rectangle(
                output,
                (x1, y1),
                (x2, y2),
                (0,255,0),
                3
            )

            cv2.putText(
                output,
                "Alligator Crack",
                (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2
            )

    return output


# Main processing loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Segment cracks
        mask = segment_alligator(gray)

        # Detect regions
        result = detect_alligator_regions(img, mask)

        # Save
        cv2.imwrite(
            os.path.join(output_folder, file),
            result
        )

        # Visualization
        if count < max_show:

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(gray, cmap="gray")
            plt.title("Enhanced Input")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(mask, cmap="gray")
            plt.title("Crack Segmentation")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            plt.title("Alligator Crack Detection")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            count += 1

print("Alligator crack detection completed successfully!")

In [ ]:
# Pothole detection

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/Pothole"
output_folder = "../../frames/segmented/Pothole"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0


# Pothole detection function
def detect_pothole(gray, output, mask):

    blur = cv2.GaussianBlur(gray, (7,7), 0)

    _, th = cv2.threshold(
        blur,
        95,
        255,
        cv2.THRESH_BINARY_INV
    )

    th = cv2.bitwise_and(th, th, mask=mask)

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_OPEN,
        np.ones((9,9), np.uint8)
    )

    th = cv2.morphologyEx(
        th,
        cv2.MORPH_CLOSE,
        np.ones((15,15), np.uint8)
    )

    contours, _ = cv2.findContours(
        th,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    count = 0

    for c in contours:

        area = cv2.contourArea(c)

        if area < 1500:
            continue

        x, y, w, h = cv2.boundingRect(c)

        if w / (h + 1e-5) > 3.5:
            continue

        cv2.rectangle(
            output,
            (x,y),
            (x+w,y+h),
            (0,0,255),
            2
        )

        cv2.putText(
            output,
            "Pothole",
            (x,y-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,0,255),
            2
        )

        count += 1

    return output


# Main processing loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)
        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Dummy full mask (since your original code expects mask)
        mask = np.ones_like(gray, dtype=np.uint8) * 255

        output = img.copy()

        # Step: pothole detection
        result = detect_pothole(gray, output, mask)

        # Save output
        cv2.imwrite(os.path.join(output_folder, file), result)

        # Visualization
        if count < max_show:

            plt.figure(figsize=(15, 5))

            plt.subplot(1, 3, 1)
            plt.imshow(gray, cmap="gray")
            plt.title("Input")
            plt.axis("off")

            plt.subplot(1, 3, 2)
            plt.imshow(mask, cmap="gray")
            plt.title("Mask")
            plt.axis("off")

            plt.subplot(1, 3, 3)
            plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            plt.title("Pothole Detection")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            count += 1

print("Pothole detection completed successfully!")

In [ ]:
# Raveling detection

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/Raveling"
output_folder = "../../frames/segmented/Raveling"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0


# Raveling detection function
def apply_enhancement(img):

    # Step 1: Smooth noise
    smoothed = cv2.blur(img, (3, 3))

    # Step 2: Logarithmic transformation
    img_float = smoothed.astype(np.float32)

    constant = 100 / np.log(
        1 + np.max(img_float)
    )

    log_img = constant * np.log(
        1 + img_float
    )

    log_img = np.array(
        log_img,
        dtype=np.uint8
    )

    # Step 3: Contrast Stretching
    low = np.min(log_img)
    high = np.max(log_img)

    if high <= low:
        return log_img

    stretched = cv2.convertScaleAbs(
        log_img,
        alpha=(255.0 / (high - low)),
        beta=-(low * 255.0 / (high - low))
    )

    return stretched

# Extract raveling damage
def extract_damage(enhanced_img):

    gaussian = cv2.GaussianBlur(
        enhanced_img,
        (5, 5),
        0
    )

    # Binarization
    binary = cv2.adaptiveThreshold(
        gaussian,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15,
        3
    )

    # Noise removal
    struct_element = np.ones(
        (3, 3),
        np.uint8
    )

    refined_mask = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        struct_element,
        iterations=2
    )

    return refined_mask


# Detect raveling regions
def detect_raveling(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for c in contours:

        if cv2.contourArea(c) > 150:

            x, y, w, h = cv2.boundingRect(c)

            cv2.rectangle(
                output,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

            cv2.putText(
                output,
                "Raveling",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    return output


# Main processing loop
for file in os.listdir(input_folder):

    if file.endswith((".jpg", ".png", ".jpeg")):

        path = os.path.join(input_folder, file)

        img = cv2.imread(path)

        if img is None:
            continue

        gray = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2GRAY
        )

        # Apply enhancement
        enhanced = apply_enhancement(gray)

        # Extract damage
        mask = extract_damage(enhanced)

        # Detect raveling
        result = detect_raveling(img, mask)

        # Save result
        cv2.imwrite(
            os.path.join(output_folder, file),
            result
        )

        # Visualization
        if count < max_show:

            plt.figure(figsize=(15,5))

            plt.subplot(1,3,1)
            plt.imshow(gray, cmap="gray")
            plt.title("Enhanced Input")
            plt.axis("off")

            plt.subplot(1,3,2)
            plt.imshow(mask, cmap="gray")
            plt.title("Raveling Mask")
            plt.axis("off")

            plt.subplot(1,3,3)
            plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
            plt.title("Detected Raveling")
            plt.axis("off")

            plt.tight_layout()
            plt.show()

            count += 1

print("Raveling detection completed successfully!")

In [ ]:
# Transverse cracking detection

# Folder structure
input_folder = "../../frames/enhanced/4_FinalEnhanced/TransverseCracking"
output_folder = "../../frames/segmented/TransverseCracking"

os.makedirs(output_folder, exist_ok=True)

max_show = 3
count = 0

# Transverse extraction function
def extract_transverse(gray):

    # CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=2.5,
        tileGridSize=(8, 8)
    )

    enhanced = clahe.apply(gray)

    # Blur
    blur = cv2.GaussianBlur(enhanced, (5, 5), 0)

    # Dark cracks become bright
    blackhat_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (21, 21)
    )

    blackhat = cv2.morphologyEx(
        blur,
        cv2.MORPH_BLACKHAT,
        blackhat_kernel
    )

    # Treshold
    _, binary = cv2.threshold(
        blackhat,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Remove small noise
    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        np.ones((3, 3), np.uint8),
        iterations=1
    )

    # Horizontal extraction 
    horizontal_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (45, 5)
    )

    horizontal = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        horizontal_kernel
    )

    # Merge nearby horizontal parts
    merge_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (35, 7)
    )

    merged = cv2.dilate(
        horizontal,
        merge_kernel,
        iterations=1
    )

    merged = cv2.morphologyEx(
        merged,
        cv2.MORPH_CLOSE,
        np.ones((9, 9), np.uint8),
        iterations=1
    )

    return merged

# Detect transverse crack regions
def detect_transverse(image, mask):

    output = image.copy()

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    H, W = image.shape[:2]

    for c in contours:

        area = cv2.contourArea(c)

        # Remove small regions
        if area < 1000:
            continue

        x, y, w, h = cv2.boundingRect(c)

        if w < 80:
            continue

        # Aspect ratio
        aspect_ratio = w / (h + 1e-5)

        if aspect_ratio < 2.5:
            continue

        # Draw detected transverse crack
        cv2.rectangle(
            output,
            (x, y),
            (x + w, y + h),
            (0, 255, 255),
            3
        )

        cv2.putText(
            output,
            "Transverse Crack",
            (x, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 255),
            2,
            cv2.LINE_AA
        )

    return output


# Main processing loop
if not os.path.isdir(input_folder):
    print(f"Input folder does not exist: {input_folder}")
else:
    for file in os.listdir(input_folder):

        if file.lower().endswith((".jpg", ".png", ".jpeg")):

            path = os.path.join(input_folder, file)

            img = cv2.imread(path)

            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            mask = extract_transverse(gray)

            result = detect_transverse(img, mask)

            # Save output
            save_path = os.path.join(output_folder, file)
            cv2.imwrite(save_path, result)

            # Visualization
            if count < max_show:

                plt.figure(figsize=(18, 6))

                plt.subplot(1, 3, 1)
                plt.imshow(gray, cmap="gray")
                plt.title("Input")
                plt.axis("off")

                plt.subplot(1, 3, 2)
                plt.imshow(mask, cmap="gray")
                plt.title("Transverse Mask")
                plt.axis("off")

                plt.subplot(1, 3, 3)
                plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
                plt.title("Detected Transverse Cracks")
                plt.axis("off")

                plt.tight_layout()
                plt.show()

                count += 1

print("Transverse crack detection completed!")